In [1]:
pip install gdown

Note: you may need to restart the kernel to use updated packages.


In [2]:
!gdown --fuzzy "https://drive.google.com/file/d/1fBiwas0OutvWk7jnwfM4x9oSosuRAe54/view?usp=sharing"

Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1fBiwas0OutvWk7jnwfM4x9oSosuRAe54

but Gdown can't. Please check connections and permissions.


In [3]:
!mkdir mri-for-brain-tumor-with-bounding-boxes
!unzip *.zip -d mri-for-brain-tumor-with-bounding-boxes

'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
!pip install -q --upgrade pip
!pip install -q --no-cache-dir torch torchvision torchaudio
!pip install -q seaborn

# Clone YOLOv5 GitHub repository
!git clone https://github.com/ultralytics/yolov5  # clone
%cd yolov5
!pip install -qr requirements.txt  # install

# Import necessary libraries
import torch
import os

ERROR: To modify pip, please run the following command:
C:\Users\Yasho\anaconda3\python.exe -m pip install -q --upgrade pip


[WinError 2] The system cannot find the file specified: 'yolov5'
C:\Users\Yasho\Downloads\BrainTumorYOLOInceptionV3-VIT-main\BrainTumorYOLOInceptionV3-VIT-main


fatal: Too many arguments.

usage: git clone [<options>] [--] <repo> [<dir>]

    -v, --[no-]verbose    be more verbose
    -q, --[no-]quiet      be more quiet
    --[no-]progress       force progress reporting
    --[no-]reject-shallow don't clone shallow repository
    -n, --no-checkout     don't create a checkout
    --checkout            opposite of --no-checkout
    --[no-]bare           create a bare repository
    --[no-]mirror         create a mirror repository (implies --bare)
    -l, --[no-]local      to clone from a local repository
    --no-hardlinks        don't use local hardlinks, always copy
    --hardlinks           opposite of --no-hardlinks
    -s, --[no-]shared     setup as shared repository
    --[no-]recurse-submodules[=<pathspec>]
                          initialize submodules in the clone
    --[no-]recursive ...  alias of --recurse-submodules
    -j, --[no-]jobs <n>   number of submodules cloned in parallel
    --[no-]template <template-directory>
            

In [5]:
import os
import shutil

# Dataset directory paths
dataset_dir = '/content/mri-for-brain-tumor-with-bounding-boxes'

# Output directories for YOLOv5
images_train_dir = '/content/yolo_data/images/train'
images_val_dir = '/content/yolo_data/images/val'
labels_train_dir = '/content/yolo_data/labels/train'
labels_val_dir = '/content/yolo_data/labels/val'

# Create directories if they do not exist
os.makedirs(images_train_dir, exist_ok=True)
os.makedirs(images_val_dir, exist_ok=True)
os.makedirs(labels_train_dir, exist_ok=True)
os.makedirs(labels_val_dir, exist_ok=True)

# Classes
classes = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

# Function to copy images and labels
def copy_files(src_image_dir, src_label_dir, dest_image_dir, dest_label_dir):
    for filename in os.listdir(src_image_dir):
        src_image_path = os.path.join(src_image_dir, filename)
        src_label_path = os.path.join(src_label_dir, os.path.splitext(filename)[0]+'.txt')
        dest_image_path = os.path.join(dest_image_dir, filename)
        dest_label_path = os.path.join(dest_label_dir, os.path.splitext(filename)[0]+'.txt')

        shutil.copy(src_image_path, dest_image_path)
        if os.path.exists(src_label_path):
            shutil.copy(src_label_path, dest_label_path)
        else:
            # Create an empty label file
            open(dest_label_path, 'a').close()

# Copy files from each class
for class_name in classes:
    # Train
    train_image_dir = os.path.join(dataset_dir, 'Train', class_name, 'images')
    train_label_dir = os.path.join(dataset_dir, 'Train', class_name, 'labels')
    copy_files(train_image_dir, train_label_dir, images_train_dir, labels_train_dir)

    # Val
    val_image_dir = os.path.join(dataset_dir, 'Val', class_name, 'images')
    val_label_dir = os.path.join(dataset_dir, 'Val', class_name, 'labels')
    copy_files(val_image_dir, val_label_dir, images_val_dir, labels_val_dir)

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/content/mri-for-brain-tumor-with-bounding-boxes\\Train\\Glioma\\images'

In [ ]:
# Create the data.yaml configuration file for YOLOv5
data_yaml = """
train: ../yolo_data/images/train
val: ../yolo_data/images/val

nc: 4  # Number of classes
names: ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']
"""

# Save the YAML config file
with open('data.yaml', 'w') as file:
    file.write(data_yaml)

In [ ]:
# Train YOLOv5 on the dataset
!python train.py --img 640 --batch 8 --epochs 10 --data data.yaml --weights yolov5s.pt --cache

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Load InceptionV3 model without the top layer
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Freeze the base model
for layer in base_model.layers:
    layer.trainable = False

# Add custom layers for classification
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(4, activation='softmax')(x)  # 4 classes

# Build the model
model = Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Data augmentation for classification
train_datagen = ImageDataGenerator(rescale=1./255, rotation_range=20, zoom_range=0.2, horizontal_flip=True)
val_datagen = ImageDataGenerator(rescale=1./255)

# Load training and validation data
train_generator = train_datagen.flow_from_directory(
    '/content/mri-for-brain-tumor-with-bounding-boxes/Train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    '/content/mri-for-brain-tumor-with-bounding-boxes/Val',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

In [ ]:
# Train the model
history = model.fit(train_generator, epochs=10, validation_data=val_generator)

In [ ]:
# Evaluate the classification model
val_loss, val_acc = model.evaluate(val_generator)
print(f"Validation Loss: {val_loss}")
print(f"Validation Accuracy: {val_acc}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Get true labels and predictions
Y_pred = model.predict(val_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = val_generator.classes

# Generate classification report
classes = list(train_generator.class_indices.keys())
print(classification_report(y_true, y_pred, target_names=classes))

# Plot confusion matrix
import seaborn as sns
import matplotlib.pyplot as plt
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
import torch

# Load the best YOLOv5 model
model = torch.hub.load('', 'custom', path='runs/train/exp/weights/best.pt', source='local')

# Get a sample image from validation set
import glob
val_images = glob.glob('/content/yolo_data/images/val/*.jpg')
img_path = val_images[0]

# Perform inference on an image
results = model(img_path)

# Display results
results.show()
results.save(save_dir='yolov5_inference_results')

In [ ]:
pip install matplotlib

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# Plot training & validation accuracy values
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('GoogleNet Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('GoogleNet Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import os
from PIL import Image

image_paths = [
    ('/content/yolov5/yolov5_inference_results15/Tr-no_0328.jpg', 'No Tumor'),
    ('/content/yolov5/yolov5_inference_results20/Tr-pi_0124.jpg', 'Pituitary'),
    ('/content/yolov5/yolov5_inference_results13/Tr-gl_0375.jpg', 'Glioma'),
    ('/content/yolov5/yolov5_inference_results14/Tr-me_0139.jpg', 'Meningioma'),
    ('/content/yolov5/yolov5_inference_results23/Tr-no_0049.jpg', 'No Tumor'),
    ('/content/yolov5/yolov5_inference_results24/Te-pi_0021.jpg', 'Pituitary'),
    ('/content/yolov5/yolov5_inference_results21/Tr-gl_0488.jpg', 'Glioma'),
    ('/content/yolov5/yolov5_inference_results22/Tr-me_1091.jpg', 'Meningioma'),
]

# Set up the plot with a 2x4 grid
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, (img_path, condition_name) in enumerate(image_paths):
    if not os.path.exists(img_path):
        print(f"Error: {img_path} does not exist.")
        continue

    # Load the image
    img = Image.open(img_path)
    img = img.resize((512, 512))

    # Determine the subplot row and column
    row = i // 4
    col = i % 4

    # Display the image in the subplot
    axes[row, col].imshow(img)
    axes[row, col].axis("off")
    axes[row, col].set_title(condition_name)

# Adjust layout and display the plot
plt.tight_layout()
plt.show()

In [ ]:
!pip install vit-keras -q

In [ ]:
!pip install tensorflow==2.15

In [ ]:
import os
import numpy as np
import tensorflow as tf
from vit_keras import vit, visualize
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, BatchNormalization
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
# Parameters
img_size = (224, 224)
batch_size = 32
data_dir = '/content/mri-for-brain-tumor-with-bounding-boxes'

# Function to preprocess images to [-1, 1] for ViT
def vit_preprocess(image, label):
    image = tf.image.resize(image, img_size)
    image = vit.preprocess_inputs(image)
    return image, label

# Load training and validation datasets from the directory structure
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    directory=f"{data_dir}/Train",
    label_mode='int',
    batch_size=batch_size,
    image_size=img_size,
    color_mode='rgb',
    shuffle=True
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    directory=f"{data_dir}/Val",
    label_mode='int',
    batch_size=batch_size,
    image_size=img_size,
    color_mode='rgb',
    shuffle=True
)

# Apply preprocessing function to datasets
train_ds = train_ds.map(vit_preprocess)
val_ds = val_ds.map(vit_preprocess)

# Check dataset structure
print(f"Training Dataset: {train_ds}")
print(f"Validation Dataset: {val_ds}")

In [ ]:
# Define the ViT model
image_size = 224
vit_base = vit.vit_b32(
    image_size=image_size,
    activation='softmax',
    pretrained=True,
    include_top=False,
    pretrained_top=False,
    classes=4
)

vit_model = Sequential([
    vit_base,
    Flatten(),
    BatchNormalization(),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dense(4, activation='softmax')  # Output layer
])

vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

vit_model.summary()

In [ ]:
# Train the model
history = vit_model.fit(
    train_ds,
    epochs=50,
    validation_data=val_ds
)

In [ ]:
# Evaluate model performance on validation set
val_loss, val_accuracy = vit_model.evaluate(val_ds)
print(f"Validation Loss: {val_loss}, Validation Accuracy: {val_accuracy}")

# Predictions and evaluation on validation data
y_true = np.concatenate([y for x, y in val_ds], axis=0)
y_pred = np.argmax(np.concatenate([vit_model.predict(x) for x, y in val_ds], axis=0), axis=1)

# Confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
class_names = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

# Plot the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='YlGnBu', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()